# Sequential Multi-Agent Workflow

This notebook creates a **sequential workflow agent** that orchestrates the two agents
from notebooks 01 and 02:

1. **enterprise-knowledge-agent** — HR policies, Bing search, AI Search (fiscal reports)
2. **med-diagnostic-agent** — LOF outlier analysis, scatter plots, treatment recommendations

Two approaches are demonstrated:
- **Approach A**: `WorkflowAgentDefinition` (declarative YAML via `azure-ai-projects` SDK)
- **Approach B**: `agent-framework` SDK with `SequentialBuilder` (pro-code)

References:
- [Foundry Workflows concept](https://learn.microsoft.com/azure/foundry/agents/concepts/workflow)
- [WorkflowAgentDefinition API](https://learn.microsoft.com/python/api/azure-ai-projects/azure.ai.projects.models.workflowagentdefinition)
- [Agent Framework sequential orchestration](https://learn.microsoft.com/agent-framework/workflows/orchestrations/sequential)
- [Agents in Workflows tutorial](https://learn.microsoft.com/agent-framework/workflows/agents-in-workflows)

### Import Libraries & Authenticate

In [1]:
import os
import json
import asyncio
from typing import Any, List, Dict
from dotenv import load_dotenv

# azure-ai-projects v2 SDK — provides AIProjectClient and agent definitions.
# WorkflowAgentDefinition is the key class for Approach A (declarative YAML).
from azure.ai.projects import AIProjectClient
import azure.ai.projects as projectslib
from azure.ai.projects.models import (
    WorkflowAgentDefinition,
    PromptAgentDefinition,
)

# OpenAI types for feeding function-call results back to the agent
from openai.types.responses.response_input_param import (
    FunctionCallOutput,
    ResponseInputParam,
)

load_dotenv(dotenv_path=".env", override=True)

from utils.fdyauth import AuthHelper
settings = AuthHelper.load_settings()
credential = AuthHelper.test_credential()

if credential:
    print('Environment and authentication OK')
else:
    print('please login first')

Environment and authentication OK


### Create Clients

In [2]:
# ---------------------------------------------------------------------------
# AIProjectClient with allow_preview=True
# ---------------------------------------------------------------------------
# WorkflowAgentDefinition requires the preview feature flag:
#   Foundry-Features: WorkflowAgents=V1Preview
# Setting allow_preview=True on the client adds this header automatically.
# Without it, create_version() returns:
#   HttpResponseError: preview_feature_required
# ---------------------------------------------------------------------------
project_client = AIProjectClient(
    credential=credential,
    endpoint=settings.project_endpoint,
    allow_preview=True,  # Required for WorkflowAgentDefinition (preview feature)
)
openai_client = project_client.get_openai_client()

print(f"azure-ai-projects version: {projectslib.__version__}")

azure-ai-projects version: 2.1.0


### Verify existing agents

The two agents must already exist from notebooks 01 and 02.
Run those notebooks first if you see errors below.

In [3]:
KNOWLEDGE_AGENT_NAME = "enterprise-knowledge-agent"
DIAGNOSTIC_AGENT_NAME = "med-diagnostic-agent"

# Verify both agents exist
knowledge_agent = project_client.agents.get(agent_name=KNOWLEDGE_AGENT_NAME)
print(f"agent > {knowledge_agent.name} (id: {knowledge_agent.id})")

diagnostic_agent = project_client.agents.get(agent_name=DIAGNOSTIC_AGENT_NAME)
print(f"agent > {diagnostic_agent.name} (id: {diagnostic_agent.id})")

agent > enterprise-knowledge-agent (id: enterprise-knowledge-agent)
agent > med-diagnostic-agent (id: med-diagnostic-agent)


---
## Approach A: Declarative Workflow (YAML + `WorkflowAgentDefinition`)

Define a sequential workflow in CSDL YAML and deploy it as a versioned agent
via `project_client.agents.create_version()` with `WorkflowAgentDefinition`.

The YAML uses `InvokeAgent` actions to call the existing agents by name.
The workflow passes a user question through:
1. The **knowledge agent** (HR / fiscal / web search context)
2. The **diagnostic agent** (patient analysis + treatment recommendation)

A router message at the start decides which path to take.

In [4]:
# ---------------------------------------------------------------------------
# Sequential Workflow YAML (CSDL format)
# ---------------------------------------------------------------------------
# Uses the trigger-based CSDL schema that the Foundry API validates.
# Requires allow_preview=True on AIProjectClient (workflow agents are preview).
# Reference: https://learn.microsoft.com/azure/foundry/agents/concepts/workflow
# ---------------------------------------------------------------------------

WORKFLOW_YAML = f"""
kind: Workflow
trigger:
  kind: OnConversationStart
  id: enterprise_diagnostic_workflow
  actions:
    - kind: InvokeAzureAgent
      id: gather_enterprise_context
      displayName: Gather enterprise context
      agent:
        name: {KNOWLEDGE_AGENT_NAME}
      output:
        autoSend: true
    - kind: InvokeAzureAgent
      id: diagnostic_analysis
      displayName: Diagnostic analysis
      agent:
        name: {DIAGNOSTIC_AGENT_NAME}
      output:
        autoSend: true
""".strip()

print(WORKFLOW_YAML)

kind: Workflow
trigger:
  kind: OnConversationStart
  id: enterprise_diagnostic_workflow
  actions:
    - kind: InvokeAzureAgent
      id: gather_enterprise_context
      displayName: Gather enterprise context
      agent:
        name: enterprise-knowledge-agent
      output:
        autoSend: true
    - kind: InvokeAzureAgent
      id: diagnostic_analysis
      displayName: Diagnostic analysis
      agent:
        name: med-diagnostic-agent
      output:
        autoSend: true


In [5]:
# ---------------------------------------------------------------------------
# Deploy the workflow as a versioned agent
# ---------------------------------------------------------------------------
# WorkflowAgentDefinition takes a single `workflow` string (the CSDL YAML).
# The API validates the YAML and creates a new agent version.
# This workflow agent can then be invoked like any other agent via
# openai_client.responses.create() with an agent_reference.
# ---------------------------------------------------------------------------
WORKFLOW_AGENT_NAME = "enterprise-diagnostic-workflow"

workflow_agent = project_client.agents.create_version(
    agent_name=WORKFLOW_AGENT_NAME,
    definition=WorkflowAgentDefinition(
        workflow=WORKFLOW_YAML,
    ),
    description="Sequential workflow: knowledge agent → diagnostic agent",
)
print(f"workflow agent > {workflow_agent.name} (id: {workflow_agent.id}, version: {workflow_agent.version})")

workflow agent > enterprise-diagnostic-workflow (id: enterprise-diagnostic-workflow:1, version: 1)


In [6]:
# ---------------------------------------------------------------------------
# Test the workflow agent via the Responses API
# ---------------------------------------------------------------------------
# The workflow agent is invoked exactly like a prompt agent:
#   - Create a conversation
#   - Call responses.create() with agent_reference pointing to the workflow
# The workflow engine executes the YAML actions sequentially:
#   1. InvokeAzureAgent -> enterprise-knowledge-agent
#   2. InvokeAzureAgent -> med-diagnostic-agent
# Each agent's output is passed to the next via autoSend.
# ---------------------------------------------------------------------------
conversation = openai_client.conversations.create()
print(f"conversation > created (id: {conversation.id})")

AGENT_REFERENCE = {"name": WORKFLOW_AGENT_NAME, "type": "agent_reference"}

response = openai_client.responses.create(
    conversation=conversation.id,
    input="Analyze patient PATIENT_ID 10 as an outlier using LOF and recommend treatment.",
    extra_body={"agent_reference": AGENT_REFERENCE},
)

print(f"\nagent > {response.output_text[:500]}...")

conversation > created (id: conv_b759fb3bf19dc50900L0FIYpFwYsfIl9YxmsK5JfokkwNRU7HK)

agent > Our company's remote work policy ensures guidelines for employees working remotely with the following key points:

- Eligibility: Employees must have completed at least 3 months and have roles that do not require constant on-site presence.
- Work Hours: Remote employees follow the same core business hours as on-site staff unless otherwise agreed.
- Communication: Use official channels like email, Slack, and virtual meetings; employees should be available during core hours.
- Data Security: Only ...


---
## Approach B: Pro-Code Workflow (`agent-framework` SDK)

Use `FoundryChatClient` + `SequentialBuilder` from the `agent-framework` package
to build a sequential workflow programmatically. This gives full control over
agent instructions, streaming, and inter-agent data passing.

> **Note**: `FoundryChatClient` requires the `agent-framework-foundry` sub-package.
> If you see `ImportError: cannot import name 'AgentMiddlewareLayer'`, upgrade:
> ```bash
> pip install --upgrade agent-framework agent-framework-foundry
> ```

References:
- [SequentialBuilder](https://learn.microsoft.com/agent-framework/workflows/orchestrations/sequential)
- [Agents in Workflows](https://learn.microsoft.com/agent-framework/workflows/agents-in-workflows)
- [Workflow as Agent](https://learn.microsoft.com/agent-framework/workflows/as-agents)

In [7]:
# ---------------------------------------------------------------------------
# Approach B: Pro-code workflow with agent-framework SDK
# ---------------------------------------------------------------------------
# FoundryChatClient wraps the same Foundry project endpoint into the
# agent-framework's Agent abstraction, allowing SequentialBuilder to
# chain agents programmatically.
#
# Known issue: agent-framework 1.2.2 has an import incompatibility with
# agent-framework-foundry on Python 3.14. If you see:
#   ImportError: cannot import name 'AgentMiddlewareLayer'
# upgrade both packages:
#   pip install --upgrade agent-framework agent-framework-foundry
# ---------------------------------------------------------------------------
from agent_framework.foundry import FoundryChatClient
from agent_framework.orchestrations import SequentialBuilder
from azure.identity import AzureCliCredential

# FoundryChatClient shares the same project endpoint and model
chat_client = FoundryChatClient(
    project_endpoint=settings.project_endpoint,
    model=settings.model_deployment_name,
    credential=AzureCliCredential(),
)

print(f"FoundryChatClient ready (model: {settings.model_deployment_name})")

FoundryChatClient ready (model: gpt-4.1-mini)


In [8]:
# Create specialized agents from the chat client
knowledge_wf_agent = chat_client.as_agent(
    name="KnowledgeAgent",
    instructions=(
        "You are an enterprise knowledge assistant at Contoso. "
        "Answer questions about HR policies, fiscal reports, and company news. "
        "If the question is about a patient or medical topic, pass it through "
        "with a note that it needs diagnostic analysis."
    ),
)

diagnostic_wf_agent = chat_client.as_agent(
    name="DiagnosticAgent",
    instructions=(
        "You are a medical diagnostic assistant. "
        "Given a patient question or context from a previous agent, "
        "analyze the patient data, run LOF outlier detection, "
        "generate a scatter plot, and recommend treatments. "
        "If the input is about enterprise/HR topics, summarize it briefly."
    ),
)

# Build sequential workflow: knowledge → diagnostic
workflow = SequentialBuilder(
    participants=[knowledge_wf_agent, diagnostic_wf_agent]
).build()

# Convert to a single workflow agent for easy invocation
workflow_as_agent = workflow.as_agent(name="Enterprise Diagnostic Pipeline")

print("Sequential workflow built: KnowledgeAgent → DiagnosticAgent")

Sequential workflow built: KnowledgeAgent → DiagnosticAgent


In [9]:
# --- Run the pro-code workflow with streaming ---
async def run_workflow():
    print("Starting workflow...")
    print("=" * 60)

    current_author = None
    async for update in workflow_as_agent.run(
        "Analyze patient PATIENT_ID 10 as an outlier using LOF and recommend treatment.",
        stream=True,
    ):
        # Show when a different agent starts responding
        if update.author_name and update.author_name != current_author:
            if current_author:
                print("\n" + "-" * 40)
            print(f"\n[{update.author_name}]:")
            current_author = update.author_name

        if update.text:
            print(update.text, end="", flush=True)

    print("\n" + "=" * 60)
    print("Workflow completed!")

await run_workflow()

Starting workflow...

[DiagnosticAgent]:
To proceed with analyzing patient PATIENT_ID 10 as an outlier using LOF (Local Outlier Factor), I need the dataset containing this patient's medical data points (e.g., lab results, vitals, symptoms, or other clinical measurements). Please provide the relevant data or upload the dataset, and I will:

1. Perform LOF outlier detection.
2. Generate a scatter plot visualizing this patient among others.
3. Recommend treatment based on the outlier analysis and clinical context.

Kindly provide the dataset or detailed patient data for further analysis.
Workflow completed!


---
### Notes

**Approach A** (declarative YAML):
- Deployed as a versioned agent via `WorkflowAgentDefinition`
- No custom code needed — YAML defines the orchestration
- Can be edited visually in the Foundry portal workflow designer
- Supports branching logic, variables, and human-in-the-loop
- The YAML format may need adjustment based on your Foundry project version

**Approach B** (pro-code `agent-framework`):
- Full programmatic control over agent instructions and data flow
- Streaming support with per-agent author attribution
- Can mix agents with custom executors (Python functions)
- Can be deployed as a hosted agent container
- Requires `agent-framework>=1.2.2` (already in requirements.txt)

**Which to choose?**
- Use **Approach A** for repeatable, no-code orchestration with portal editing
- Use **Approach B** when you need custom logic, streaming, or local testing

### (Optional) Cleanup

In [10]:
# # Delete the workflow agent version
# project_client.agents.delete_version(
#     agent_name=workflow_agent.name,
#     agent_version=workflow_agent.version,
# )
# print("Workflow agent version deleted.")